<a href="https://colab.research.google.com/github/Alister44/melbourne-housing-price-prediction/blob/main/housing_price_prediction_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
RANDOM_STATE = 42

In [2]:
FILE_IN = 'https://raw.githubusercontent.com/Alister44/melbourne-housing-price-prediction/refs/heads/main/data/melb_data_raw.csv'
df_raw = pd.read_csv(FILE_IN)
df = df_raw.copy()

In [3]:
# drop the technical index column if it exists
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

In [4]:
pd.set_option("display.max_columns", None)
df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,3.0,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


Analysis of missing values and types

In [5]:
def info_types_missVals():
    return pd.DataFrame({
        "dtype": df.dtypes,
        "missing count": df.isnull().sum(),
        "missing (%)": (df.isna().mean() * 100).round(2)
    }).sort_values("missing (%)", ascending=False)

info_types_missVals()

,dtype,missing count,missing (%)
BuildingArea,float64,6450,47.50
YearBuilt,float64,5375,39.58
CouncilArea,object,1369,10.08
Car,float64,62,0.46
Suburb,object,0,0.00
Price,float64,0,0.00
Type,object,0,0.00
Rooms,int64,0,0.00
Address,object,0,0.00
Distance,float64,0,0.00


In [6]:
df['Date'] = pd.to_datetime(df['Date'],errors='coerce', dayfirst=True)
df['YearBuilt'] = df['YearBuilt'].astype('Int32')
df['Rooms'] = df['Rooms'].astype('Int8')
df['Bedroom2'] = df['Bedroom2'].astype('Int8')
df['Bathroom'] = df['Bathroom'].astype('Int8')
df['Car'] = df['Car'].astype('Int8')
df['Propertycount'] = df['Propertycount'].astype('Int64')

In [7]:
info_types_missVals()

,dtype,missing count,missing (%)
BuildingArea,float64,6450,47.50
YearBuilt,Int32,5375,39.58
CouncilArea,object,1369,10.08
Car,Int8,62,0.46
Suburb,object,0,0.00
Price,float64,0,0.00
Type,object,0,0.00
Rooms,Int8,0,0.00
Address,object,0,0.00
Distance,float64,0,0.00


In [8]:
df.describe() # descriptive statistics of numeric columns

,Rooms,Price,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.0,1.358000e+04,13580,13580.000000,13580.000000,13580.0,13580.0,13518.0,13580.000000,7130.000000,8205.0,13580.000000,13580.000000,13580.0
mean,2.937997,1.075684e+06,2017-01-28 20:25:03.622974976,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
min,1.0,8.500000e+04,2016-01-28 00:00:00,0.000000,3000.000000,0.0,0.0,0.0,0.000000,0.000000,1196.0,-38.182550,144.431810,249.0
25%,2.0,6.500000e+05,2016-09-10 00:00:00,6.100000,3044.000000,2.0,1.0,1.0,177.000000,93.000000,1940.0,-37.856822,144.929600,4380.0
50%,3.0,9.030000e+05,2017-03-04 00:00:00,9.200000,3084.000000,3.0,1.0,2.0,440.000000,126.000000,1970.0,-37.802355,145.000100,6555.0
75%,3.0,1.330000e+06,2017-06-24 00:00:00,13.000000,3148.000000,3.0,2.0,2.0,651.000000,174.000000,1999.0,-37.756400,145.058305,10331.0
max,10.0,9.000000e+06,2017-09-23 00:00:00,48.100000,3977.000000,20.0,8.0,10.0,433014.000000,44515.000000,2018.0,-37.408530,145.526350,21650.0
std,0.955748,6.393107e+05,NaN,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772


In [9]:
df.describe(include='all') # descriptive statistics for all columns

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
count,13580,13580,13580.0,13580,1.358000e+04,13580,13580,13580,13580.000000,13580.000000,13580.0,13580.0,13518.0,13580.000000,7130.000000,8205.0,12211,13580.000000,13580.000000,13580,13580.0
unique,314,13378,<NA>,3,NaN,5,268,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,<NA>,33,NaN,NaN,8,<NA>
top,Reservoir,5 Margaret St,<NA>,h,NaN,S,Nelson,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,<NA>,Moreland,NaN,NaN,Southern Metropolitan,<NA>
freq,359,3,<NA>,9449,NaN,9022,1565,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,<NA>,1163,NaN,NaN,4695,<NA>
mean,NaN,NaN,2.937997,NaN,1.075684e+06,NaN,NaN,2017-01-28 20:25:03.622974976,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,NaN,-37.809203,144.995216,NaN,7454.417378
min,NaN,NaN,1.0,NaN,8.500000e+04,NaN,NaN,2016-01-28 00:00:00,0.000000,3000.000000,0.0,0.0,0.0,0.000000,0.000000,1196.0,NaN,-38.182550,144.431810,NaN,249.0
25%,NaN,NaN,2.0,NaN,6.500000e+05,NaN,NaN,2016-09-10 00:00:00,6.100000,3044.000000,2.0,1.0,1.0,177.000000,93.000000,1940.0,NaN,-37.856822,144.929600,NaN,4380.0
50%,NaN,NaN,3.0,NaN,9.030000e+05,NaN,NaN,2017-03-04 00:00:00,9.200000,3084.000000,3.0,1.0,2.0,440.000000,126.000000,1970.0,NaN,-37.802355,145.000100,NaN,6555.0
75%,NaN,NaN,3.0,NaN,1.330000e+06,NaN,NaN,2017-06-24 00:00:00,13.000000,3148.000000,3.0,2.0,2.0,651.000000,174.000000,1999.0,NaN,-37.756400,145.058305,NaN,10331.0
max,NaN,NaN,10.0,NaN,9.000000e+06,NaN,NaN,2017-09-23 00:00:00,48.100000,3977.000000,20.0,8.0,10.0,433014.000000,44515.000000,2018.0,NaN,-37.408530,145.526350,NaN,21650.0



##  Data Preparation & Cleaning (Python)

### Note to `Method`
The table contains only five sale methods: S, SP, PI, VB and SA.
Only S, SP and SA clearly mean that the sale took place.

- **PI (passed in):** the auction took place, but the property was not sold.
  However, PI rows have a value in the `Date` column (date sold), which is
  a possible contradiction.
- **VB (vendor bid):** the outcome of the deal is not clearly stated, so the
  property may not have been sold. In that case, `Price` may contain the last
  or highest bid rather than the final sale price.

**Decision:** as an analyst, I exclude properties with the ambiguous statuses
PI and VB from the model. The goal of the model is to predict the sale price.
If `Price` for PI and VB may not be the final price, these rows add noise to
the target variable, and the model learns from inaccurate answers.

In [10]:
df['Method'].unique()

array(['S', 'SP', 'PI', 'VB', 'SA'], dtype=object)

In [11]:
df["Method"].value_counts(normalize=True)*100

,proportion
Method,
S,66.435935
SP,12.540501
PI,11.516937
VB,8.829161
SA,0.677467


In [12]:
rows_before = len(df)
rows_before

13580

In [13]:
df = df[~df["Method"].isin(["PI", "VB"])].copy()
rows_removed = rows_before - len(df)

print(f"\nRemoved {rows_removed} rows ({rows_removed / rows_before:.1%} of the data)")
print(f"Rows left: {len(df)}")

# Check: only S, SP and SA should remain
print(df["Method"].unique())


Removed 2763 rows (20.3% of the data)
Rows left: 10817
['S' 'SP' 'SA']


### Note on `Bedroom2`
In 1.5% of rows `Bedroom2` is greater than `Rooms`. `Bedroom2` comes from
a different source, so the difference may reflect different definitions
rather than errors. Since the two columns are almost identical, I kept
`Rooms` and dropped `Bedroom2`. I did not remove any rows.

In [14]:
(df["Bedroom2"] > df["Rooms"]).sum() # number of objects where number of bedrooms more that totaly rooms

np.int64(161)

In [15]:
(df["Bedroom2"] > df["Rooms"]).mean() * 100 # number of objects where number of bedrooms more that totaly rooms

np.float64(1.4883978922067116)

In [16]:
df = df.drop(columns=["Bedroom2"])

### Note on `Bathroom`
In 28 rows (0.2%) `Bathroom` is greater than `Rooms`. In 23 of them the
difference is 1, which is unusual but possible. In 5 rows the difference is
2 or more. I did not change these rows, because the effect on the model is
negligible and I cannot tell which of the two columns is wrong.

In [17]:
(df["Bathroom"] > df["Rooms"]).mean() * 100 # number of objects where number of Bathroom more that totaly rooms

np.float64(0.18489414810021262)

In [18]:
(df["Bathroom"] > df["Rooms"]).sum()

np.int64(20)

In [19]:

(df["Bathroom"] - df["Rooms"])[df["Bathroom"] > df["Rooms"]].value_counts()

,count
1,17
3,2
2,1


Note to `Car`  
`Car` has 0.43% missing values. I left them unchanged in the cleaning step;
they are imputed with the median inside the pipeline, fitted on the
training data only.

In [20]:
info_types_missVals()

,dtype,missing count,missing (%)
BuildingArea,float64,5193,48.01
YearBuilt,Int32,4360,40.31
CouncilArea,object,1089,10.07
Car,Int8,43,0.40
Type,object,0,0.00
Rooms,Int8,0,0.00
Address,object,0,0.00
Suburb,object,0,0.00
Date,datetime64[ns],0,0.00
Price,float64,0,0.00


### Note to `Landsize`
I treat `Landsize` as the plot area of the property, in square meters.
I kept zeros. In Australia it is common for houses, townhouses and duplexes
to come without a private plot of land, so 0 is treated as a real value, not
as missing data. Zeros make up 1.8% of houses, 12.5% of townhouses and 54%
of units.

In [21]:
zero_share = (df["Landsize"] == 0).groupby(df["Type"]).mean() * 100 # Share of properties with Landsize = 0 within each property type (%).
print(zero_share.round(1))

Type
h     1.7
t    13.2
u    51.5
Name: Landsize, dtype: float64


### `BuildingArea`
An area of 0 is not a real measurement, so I replaced X zeros with missing
values. I also replaced 60 values below 15 m² for properties with more than
one room, which is not realistic. Properties with one room and an area of
10-15 m² were kept, because a studio of that size is unusual but possible.
All missing values are imputed with the median in the pipeline.

In [22]:
# A zero building area is not a real value, so I treat it as missing.
# Landsize = 0 is kept, because a property can have no land of its own.
df['BuildingArea'] = df['BuildingArea'].replace(0,pd.NA)

In [23]:
# A building area below 15 m² is not realistic for a property with more
# than one room, so I treat it as a data error and set it to missing.
mask = (df["BuildingArea"] < 15) & (df["Rooms"] > 1)
print(mask.sum())   # how many values are affected
df.loc[mask, "BuildingArea"] = np.nan

41


### Note to `YearBuilt`

We check whether the YearBuilt is specified correctly and display a few rows in ascending order. We see that 1196 is clearly an error. Next comes 1830, which looks more plausible.

In [24]:
df['YearBuilt'].sort_values().head()

,YearBuilt
2079,1830
2554,1850
4843,1850
5405,1850
2530,1854


In [25]:
df.loc[df['YearBuilt'] == 1196, 'YearBuilt'] = pd.NA # replaced "1196" with NA
df['YearBuilt'].sort_values(ascending=False).head()#checked for any erroneous maximum date

,YearBuilt
1234,2018
8134,2017
12629,2017
5153,2017
4282,2017


Duplicate check:

In [27]:
print(f"Duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Duplicates after removal: {df.duplicated().sum()}")

Duplicates: 0
Duplicates after removal: 0
